In [1]:
from pathlib import Path
import gcamreader
import os
import pandas as pd
import numpy as np
from utils import convert_to_mt

In [2]:
dfCO2Map = pd.read_csv("./extdata/gcamreport/CO2_tech_map.csv", skiprows=[0])
dfCO2Map.head()

,sector,subsector,technology,var1,var2,var3,var4,var5,var6,var7,var8,var9,unit_conv
0,airCO2,airCO2,airCO2,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
1,CO2 removal,dac,hightemp DAC NG,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
2,CO2 removal,dac,hightemp DAC elec,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
3,CO2 removal,dac,lowtemp DAC heatpump,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
4,agricultural energy use,mobile,refined liquids,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Demand,Emissions|CO2|Energy|Demand|AFOFI,Emissions|CO2|Energy|Demand|Residential and Co...,NaN,NaN,NaN,3.666667


In [3]:
dfNonCo2Map = pd.read_csv("./extdata/gcamreport/nonCO2_emissions_sector_map.csv", skiprows=[0])
dfNonCo2Map

,sector,subsector,ghg,var1,var2,var3,var4,var5,var6,var7,var8,unit_conv
0,agricultural energy use,NaN,BC,Emissions|BC,Emissions|BC|Energy,Emissions|BC|Energy|Demand,Emissions|BC|Energy|Demand|AFOFI,NaN,NaN,Emissions|BC|Energy|Demand|Residential and Com...,Emissions|BC|Energy and Industrial Processes,1.0
1,agricultural energy use,NaN,CH4,Emissions|CH4,Emissions|CH4|Energy,Emissions|BC|Energy|Demand,Emissions|CH4|Energy|Demand|AFOFI,NaN,NaN,Emissions|CH4|Energy|Demand|Residential and Co...,Emissions|CH4|Energy and Industrial Processes,1.0
2,agricultural energy use,NaN,CO,Emissions|CO,Emissions|CO|Energy,Emissions|BC|Energy|Demand,Emissions|CO|Energy|Demand|AFOFI,NaN,NaN,Emissions|CO|Energy|Demand|Residential and Com...,Emissions|CO|Energy and Industrial Processes,1.0
3,agricultural energy use,NaN,N2O,Emissions|N2O,Emissions|N2O|Energy,Emissions|N2O|Energy|Demand,Emissions|N2O|Energy|Demand|AFOFI,NaN,NaN,Emissions|N2O|Energy|Demand|Residential and Co...,Emissions|N2O|Energy and Industrial Processes,1000.0
4,agricultural energy use,NaN,NH3,Emissions|NH3,Emissions|NH3|Energy,Emissions|NH3|Energy|Demand,Emissions|NH3|Energy|Demand|AFOFI,NaN,NaN,Emissions|NH3|Energy|Demand|Residential and Co...,Emissions|NH3|Energy and Industrial Processes,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
918,urban processes,NaN,OC,Emissions|OC,Emissions|OC|Other,NaN,NaN,NaN,NaN,NaN,NaN,1.0
919,urban processes,NaN,SO2_1,Emissions|Sulfur,Emissions|Sulfur|Other,NaN,NaN,NaN,NaN,NaN,NaN,1.0
920,urban processes,NaN,SO2_2,Emissions|Sulfur,Emissions|Sulfur|Other,NaN,NaN,NaN,NaN,NaN,NaN,1.0
921,urban processes,NaN,SO2_3,Emissions|Sulfur,Emissions|Sulfur|Other,NaN,NaN,NaN,NaN,NaN,NaN,1.0


In [88]:
def to_Mt(row):
    val, unit = row['value'], row['Units']
    if unit == 'Tg':
        return val
    elif unit == 'Gg':
        return val * 1e-3
    elif unit == 'MTC':
        return val * (44.009 / 12.011)
    else:
        raise ValueError(f"Unknown unit: {unit}")

# AR5 100-yr GWP
GWP_AR5 = {
    'CO2':    1,
    'CH4':    28,
    'N2O':    265,
    'HFC125': 3500,
    'HFC134a':1430,
    'HFC143a':4470,
    'HFC23':  14800,
    'HFC32':  675,
    'HFC43':  1500,
    'HFC227ea':3220,
    'HFC236fa':9810,
    'SF6':    23500,
    'C2F6':   12200,
    'CF4':    6630,
}

In [89]:
proj_path = Path("/data/project/tae/gcam-core")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

In [4]:
dbpath = "../output/"  # relative to current working directory
dbfile = "database_basexdb_korea_2035_20250701"
conn = gcamreader.LocalDBConn(dbpath, dbfile)
queries = gcamreader.parse_batch_query(os.path.join('..', 'output', 'queries','Main_queries.xml'))

Database scenarios: Enhanced-Ambition, Current-Policy, Enhanced-Ambition-Bld


In [5]:
scenarios = list(conn.listScenariosInDB()['name'])
scenarios

['Enhanced-Ambition', 'Current-Policy', 'Enhanced-Ambition-Bld']

In [6]:
for i, q in enumerate(queries):
    print(i, q.title)

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [7]:
q = queries[314]
print(q.title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df

prices of all markets


,Units,scenario,Year,market,value
0,$/GJ,Current-Policy,1975,South Koreawoodpulp_energy,0.82642
1,$/GJ,Current-Policy,1990,South Koreawoodpulp_energy,98.45780
2,$/GJ,Current-Policy,2005,South Koreawoodpulp_energy,51.64210
3,$/GJ,Current-Policy,2010,South Koreawoodpulp_energy,45.54300
4,$/GJ,Current-Policy,2015,South Koreawoodpulp_energy,26.85570
...,...,...,...,...,...
24217,unitless,Enhanced-Ambition-Bld,2090,South KoreaFoodDemand_Staples-budget-fraction-...,0.10000
24218,unitless,Enhanced-Ambition-Bld,2095,South KoreaFoodDemand_NonStaples-budget-fracti...,0.10000
24219,unitless,Enhanced-Ambition-Bld,2095,South KoreaFoodDemand_Staples-budget-fraction-...,0.10000
24220,unitless,Enhanced-Ambition-Bld,2100,South KoreaFoodDemand_NonStaples-budget-fracti...,0.10000


In [113]:
df['market'].unique()

array(['South Koreawoodpulp_energy', 'South Koreasawnwood_processing',
       'South Koreawoodpulp_processing',
       'South KoreaH2 central production', 'South KoreaH2 industrial',
       'South KoreaH2 liquid truck', 'South KoreaH2 pipeline',
       'South KoreaH2 retail delivery', 'South KoreaH2 retail dispensing',
       'South KoreaH2 wholesale delivery',
       'South KoreaH2 wholesale dispensing',
       'South Koreaagricultural energy use',
       'South Koreabackup_electricity', 'South Koreabiomass',
       'South Koreachemical', 'South Koreachemical energy use',
       'South Koreachemical feedstocks', 'South Koreacoal',
       'South Koreacomm cooling', 'South Koreacomm heating',
       'South Koreacomm others', 'South Koreaconstruction',
       'South Koreaconstruction energy use',
       'South Koreaconstruction feedstocks', 'South Koreacrude oil',
       'South Koreacsp_backup', 'South Koreadelivered biomass',
       'South Koreadelivered coal', 'South Koreadelivered gas

In [108]:
df['scenario'].unique()

array(['Test'], dtype=object)

In [114]:
df[(df['market'] == 'South Koreaelectricity')]

,Units,scenario,Year,market,value
115,1975$/GJ,Test,1975,South Koreaelectricity,5.25070
272,1975$/GJ,Test,1990,South Koreaelectricity,8.41542
429,1975$/GJ,Test,2005,South Koreaelectricity,7.79794
586,1975$/GJ,Test,2010,South Koreaelectricity,8.25390
743,1975$/GJ,Test,2015,South Koreaelectricity,8.21538
900,1975$/GJ,Test,2020,South Koreaelectricity,8.18610
1057,1975$/GJ,Test,2025,South Koreaelectricity,8.17284
1214,1975$/GJ,Test,2030,South Koreaelectricity,7.95865
1371,1975$/GJ,Test,2035,South Koreaelectricity,7.79320
1528,1975$/GJ,Test,2040,South Koreaelectricity,1.00000


In [111]:
df[(df['Year'] == 2035) & (df['sector'] == 'resid heating modern_d2')]

,Units,scenario,region,sector,subsector,Year,value
1328,none,Test,South Korea,resid heating modern_d2,biomass,2035,0.000000
1350,none,Test,South Korea,resid heating modern_d2,electricity,2035,1.793000
1372,none,Test,South Korea,resid heating modern_d2,gas,2035,0.333333
1394,none,Test,South Korea,resid heating modern_d2,refined liquids,2035,0.613782


In [637]:
df[(df['Year'] >= 2035)]

,Units,scenario,region,sector,subsector,Year,value
8,none,Reference,South Korea,comm cooling,electricity,2035,1.00000
9,none,Reference,South Korea,comm cooling,electricity,2040,1.00000
10,none,Reference,South Korea,comm cooling,electricity,2045,1.00000
11,none,Reference,South Korea,comm cooling,electricity,2050,1.00000
12,none,Reference,South Korea,comm cooling,electricity,2055,1.00000
...,...,...,...,...,...,...,...
3339,none,Reference,South Korea,resid others modern_d9,refined liquids,2080,1.82008
3340,none,Reference,South Korea,resid others modern_d9,refined liquids,2085,1.82008
3341,none,Reference,South Korea,resid others modern_d9,refined liquids,2090,1.82008
3342,none,Reference,South Korea,resid others modern_d9,refined liquids,2095,1.82008


In [373]:
df['subsector'].unique()

KeyError: 'subsector'

In [371]:
df[(df['subsector'] == 'Bus') & (df['Year'] == 2030)]

,Units,scenario,region,sector,subsector,technology,Year,value
78,million pass-km,Current-Policy,South Korea,trn_pass_road,Bus,"BEV,year=2030",2030,64405.30
82,million pass-km,Current-Policy,South Korea,trn_pass_road,Bus,"FCEV,year=2030",2030,7981.67
86,million pass-km,Current-Policy,South Korea,trn_pass_road,Bus,"Hybrid Liquids,year=2030",2030,198011.00
95,million pass-km,Current-Policy,South Korea,trn_pass_road,Bus,"Liquids,year=2030",2030,203041.00
102,million pass-km,Current-Policy,South Korea,trn_pass_road,Bus,"NG,year=2030",2030,33157.30
344,million pass-km,Enhanced-Ambition,South Korea,trn_pass_road,Bus,"BEV,year=2030",2030,75862.40
348,million pass-km,Enhanced-Ambition,South Korea,trn_pass_road,Bus,"FCEV,year=2030",2030,7413.84
352,million pass-km,Enhanced-Ambition,South Korea,trn_pass_road,Bus,"Hybrid Liquids,year=2030",2030,184068.00
361,million pass-km,Enhanced-Ambition,South Korea,trn_pass_road,Bus,"Liquids,year=2030",2030,188504.00
368,million pass-km,Enhanced-Ambition,South Korea,trn_pass_road,Bus,"NG,year=2030",2030,30826.20


In [359]:
df[(df['Year'] == 2035)]

,Units,scenario,region,input,Year,value
3,EJ,Current-Policy,South Korea,H2 retail dispensing,2035,0.012605
6,EJ,Current-Policy,South Korea,H2 wholesale dispensing,2035,0.000525
14,EJ,Current-Policy,South Korea,delivered gas,2035,0.020290
23,EJ,Current-Policy,South Korea,elect_td_trn,2035,0.075954
32,EJ,Current-Policy,South Korea,refined liquids enduse,2035,1.845156
36,EJ,Enhanced-Ambition,South Korea,H2 retail dispensing,2035,0.012145
39,EJ,Enhanced-Ambition,South Korea,H2 wholesale dispensing,2035,0.000488
47,EJ,Enhanced-Ambition,South Korea,delivered gas,2035,0.018449
56,EJ,Enhanced-Ambition,South Korea,elect_td_trn,2035,0.073730
65,EJ,Enhanced-Ambition,South Korea,refined liquids enduse,2035,1.747873


In [8]:
q = queries[262]
print(q.title)
dfCO2 = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
dfCO2['scenario'] = dfCO2['scenario'].str.split(',').str[0]
dfCO2['GHG'] = 'CO2'

CO2 emissions by sector (no bio) (excluding resource production)


In [9]:
# removes _d1 … _d10 only when they appear at the very end of the string
dfCO2['sector'] = dfCO2['sector'].str.replace(r'_d(?:[1-9]|10)$', '', regex=True)

In [10]:
for sec in dfCO2['sector'].unique():
    if sec not in dfCO2Map['sector'].unique():
        print(sec)

electricity


In [11]:
dfCO2Map.head()

,sector,subsector,technology,var1,var2,var3,var4,var5,var6,var7,var8,var9,unit_conv
0,airCO2,airCO2,airCO2,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
1,CO2 removal,dac,hightemp DAC NG,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
2,CO2 removal,dac,hightemp DAC elec,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
3,CO2 removal,dac,lowtemp DAC heatpump,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
4,agricultural energy use,mobile,refined liquids,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Demand,Emissions|CO2|Energy|Demand|AFOFI,Emissions|CO2|Energy|Demand|Residential and Co...,NaN,NaN,NaN,3.666667


In [12]:
dfCO2

,Units,scenario,region,sector,Year,value,GHG
0,MTC,Current-Policy,South Korea,H2 central production,2020,3.094979e-03,CO2
1,MTC,Current-Policy,South Korea,H2 central production,2025,7.159770e-03,CO2
2,MTC,Current-Policy,South Korea,H2 central production,2030,6.211017e-03,CO2
3,MTC,Current-Policy,South Korea,H2 central production,2035,9.004936e-03,CO2
4,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2020,2.875939e-03,CO2
...,...,...,...,...,...,...,...
1785,MTC,Enhanced-Ambition-Bld,South Korea,wholesale gas,2010,-3.020223e-07,CO2
1786,MTC,Enhanced-Ambition-Bld,South Korea,wholesale gas,2015,-8.339991e-08,CO2
1787,MTC,Enhanced-Ambition-Bld,South Korea,wholesale gas,2020,-7.261928e-08,CO2
1788,MTC,Enhanced-Ambition-Bld,South Korea,wholesale gas,2030,-1.880560e-07,CO2


In [13]:
dfCO2Sec = dfCO2.merge(dfCO2Map[['sector', 'var1', 'var2', 'var3', 'var4', 'var5']].drop_duplicates(), on=['sector'], how='left')
dfCO2Sec.head()

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5
0,MTC,Current-Policy,South Korea,H2 central production,2020,0.003095,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen
1,MTC,Current-Policy,South Korea,H2 central production,2025,0.007160,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen
2,MTC,Current-Policy,South Korea,H2 central production,2030,0.006211,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen
3,MTC,Current-Policy,South Korea,H2 central production,2035,0.009005,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen
4,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2020,0.002876,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen


In [14]:
dfCO2Sec[dfCO2Sec['var1'].isna()]

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5
117,MTC,Current-Policy,South Korea,electricity,1990,9.388615,CO2,NaN,NaN,NaN,NaN,NaN
118,MTC,Current-Policy,South Korea,electricity,2005,47.378742,CO2,NaN,NaN,NaN,NaN,NaN
119,MTC,Current-Policy,South Korea,electricity,2010,67.091207,CO2,NaN,NaN,NaN,NaN,NaN
120,MTC,Current-Policy,South Korea,electricity,2015,69.163555,CO2,NaN,NaN,NaN,NaN,NaN
121,MTC,Current-Policy,South Korea,electricity,2020,64.789681,CO2,NaN,NaN,NaN,NaN,NaN
122,MTC,Current-Policy,South Korea,electricity,2025,55.857738,CO2,NaN,NaN,NaN,NaN,NaN
123,MTC,Current-Policy,South Korea,electricity,2030,43.248456,CO2,NaN,NaN,NaN,NaN,NaN
124,MTC,Current-Policy,South Korea,electricity,2035,27.837278,CO2,NaN,NaN,NaN,NaN,NaN
712,MTC,Enhanced-Ambition,South Korea,electricity,1990,9.388615,CO2,NaN,NaN,NaN,NaN,NaN
713,MTC,Enhanced-Ambition,South Korea,electricity,2005,47.378742,CO2,NaN,NaN,NaN,NaN,NaN


In [15]:
dfCO2Sec['var5'].unique()

array(['Emissions|CO2|Energy|Supply|Hydrogen',
       'Emissions|CO2|Energy|Demand|AFOFI',
       'Emissions|CO2|Energy|Demand|Industry',
       'Emissions|CO2|Energy|Supply|Electricity', nan,
       'Emissions|CO2|Energy|Demand|Residential and Commercial',
       'Emissions|CO2|Energy|Supply|Solids',
       'Emissions|CO2|Energy|Supply|Gases',
       'Emissions|CO2|Energy|Demand|Other Sector',
       'Emissions|CO2|Energy|Supply|Liquids',
       'Emissions|CO2|Energy|Demand|Transportation'], dtype=object)

In [16]:
def cat_sec(row):
    if row['sector'] == 'electricity':
        return 'Electricity'
    elif row['sector'] in ['airCO2', 'CO2 removal']:
        return 'Absortion / Removal'
    elif row['sector'] == 'cement':
        return 'Industry'
    elif row['sector'] == 'desalinated water':
        return 'Buildings'
    elif row['var5'].endswith('Hydrogen'):
        return 'Hydrogen'
    elif row['var5'].endswith('AFOFI'):
        return 'AFOFI'
    elif row['var5'].endswith('Industry'):
        return 'Industry'
    elif row['var5'].endswith('Electricity'):
        return "Electricity"
    elif row['var5'].endswith('Residential and Commercial'):
        return "Buildings"
    elif row['var5'].endswith('Transportation'):
        return 'Transportation'
    elif row['sector'] in ['delivered biomass', 'delivered gas', 'gas pipeline', 'gas processing', 'refined liquids enduse', 'refined liquids industrial', 'refining', 'wholesale gas']:
        return 'Industry'
    else:
        print(row['sector'])
        return "Others"

In [17]:
dfCO2Sec['sec'] = dfCO2Sec.apply(cat_sec, axis=1)

In [18]:
dfCO2Sec

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5,sec
0,MTC,Current-Policy,South Korea,H2 central production,2020,3.094979e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Hydrogen
1,MTC,Current-Policy,South Korea,H2 central production,2025,7.159770e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Hydrogen
2,MTC,Current-Policy,South Korea,H2 central production,2030,6.211017e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Hydrogen
3,MTC,Current-Policy,South Korea,H2 central production,2035,9.004936e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Hydrogen
4,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2020,2.875939e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Hydrogen
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1785,MTC,Enhanced-Ambition-Bld,South Korea,wholesale gas,2010,-3.020223e-07,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Gases,Industry
1786,MTC,Enhanced-Ambition-Bld,South Korea,wholesale gas,2015,-8.339991e-08,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Gases,Industry
1787,MTC,Enhanced-Ambition-Bld,South Korea,wholesale gas,2020,-7.261928e-08,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Gases,Industry
1788,MTC,Enhanced-Ambition-Bld,South Korea,wholesale gas,2030,-1.880560e-07,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Gases,Industry


In [19]:
scenarios

['Enhanced-Ambition', 'Current-Policy', 'Enhanced-Ambition-Bld']

In [21]:
dfCO2Sec[(dfCO2Sec['Year'].isin([2020, 2035]) & (~dfCO2Sec['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl']))) & (dfCO2Sec['scenario'].isin(['Current-Policy', 'Enhanced-Ambition', 'Enhanced-Ambition-Bld']))].groupby(['scenario', 'Year', 'sec'])['value'].sum()

scenario               Year  sec                
Current-Policy         2020  AFOFI                   0.381115
                             Buildings              14.053269
                             Electricity            64.836859
                             Hydrogen                0.005971
                             Industry               55.529424
                             Transportation         31.888004
                       2035  AFOFI                   0.317919
                             Buildings              13.992602
                             Electricity            29.173632
                             Hydrogen                0.081383
                             Industry               51.613119
                             Transportation         28.799149
Enhanced-Ambition      2020  AFOFI                   0.381165
                             Buildings              14.051681
                             Electricity            64.834213
                     

In [70]:
35/55

0.6363636363636364

In [71]:
dfCO2Sec[(dfCO2Sec['sec'] == 'Electricity')]['sector'].unique()

array(['backup_electricity', 'elec_coal (IGCC CCS)',
       'elec_coal (conv pul CCS)', 'elec_gas (CC CCS)', 'electricity'],
      dtype=object)

In [72]:
dfCO2Sec[(dfCO2Sec['Year'].isin([2020, 2035])) & (dfCO2Sec['scenario'].isin(['Current-Policy', 'Enhanced-Ambition']))].groupby(['scenario', 'Year', 'sector'])['value'].sum().reset_index()#.pivot(index=['scenario', 'sec'], columns=['Year'], values='value')

,scenario,Year,sector,value
0,Current-Policy,2020,H2 central production,0.003095
1,Current-Policy,2020,H2 wholesale dispensing,0.002876
2,Current-Policy,2020,agricultural energy use,0.381115
3,Current-Policy,2020,ammonia,0.219538
4,Current-Policy,2020,backup_electricity,0.047178
...,...,...,...,...
163,Enhanced-Ambition,2035,trn_pass_road,5.117367
164,Enhanced-Ambition,2035,trn_pass_road_LDV,0.109964
165,Enhanced-Ambition,2035,trn_pass_road_LDV_4W,7.413888
166,Enhanced-Ambition,2035,trn_shipping_intl,6.722328


In [73]:
dfCO2Sec[(dfCO2Sec['Year'].isin([2020, 2035])) & (dfCO2Sec['sec'].isin(['Buildings']))]

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5,sec
57,MTC,Current-Policy,South Korea,comm cooling,2020,0.141053,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Demand,Emissions|CO2|Energy|Demand|Residential and Co...,Buildings
60,MTC,Current-Policy,South Korea,comm cooling,2035,0.091915,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Demand,Emissions|CO2|Energy|Demand|Residential and Co...,Buildings
65,MTC,Current-Policy,South Korea,comm heating,2020,3.381331,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Demand,Emissions|CO2|Energy|Demand|Residential and Co...,Buildings
68,MTC,Current-Policy,South Korea,comm heating,2035,3.827472,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Demand,Emissions|CO2|Energy|Demand|Residential and Co...,Buildings
73,MTC,Current-Policy,South Korea,comm others,2020,2.132568,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Demand,Emissions|CO2|Energy|Demand|Residential and Co...,Buildings
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1091,MTC,Enhanced-Ambition,South Korea,resid others modern,2035,0.260960,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Demand,Emissions|CO2|Energy|Demand|Residential and Co...,Buildings
1096,MTC,Enhanced-Ambition,South Korea,resid others modern,2020,0.295594,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Demand,Emissions|CO2|Energy|Demand|Residential and Co...,Buildings
1099,MTC,Enhanced-Ambition,South Korea,resid others modern,2035,0.277585,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Demand,Emissions|CO2|Energy|Demand|Residential and Co...,Buildings
1104,MTC,Enhanced-Ambition,South Korea,resid others modern,2020,0.332251,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Demand,Emissions|CO2|Energy|Demand|Residential and Co...,Buildings


In [330]:
dfDiff = dfCO2Sec[(dfCO2Sec['Year'].isin([2020, 2035])) & (dfCO2Sec['scenario'].isin(['Current-Policy', 'Enhanced-Ambition']))].groupby(['scenario', 'Year', 'sec'])['value'].sum().reset_index().pivot(index=['scenario', 'sec'], columns=['Year'], values='value')
dfDiff['diff'] = dfDiff[2020] - dfDiff[2035]
dfDiff['rr'] = (dfDiff['diff'] / dfDiff[2020]) * 100
dfDiff

Year                                        2020       2035       diff  \
scenario          sec                                                    
Current-Policy    AFOFI                 0.381310   0.319466   0.061844   
                  Buildings            13.617009  13.552398   0.064611   
                  Electricity          65.007465  29.235749  35.771716   
                  Hydrogen              0.006012   0.083267  -0.077255   
                  Industry             55.490154  51.769758   3.720395   
                  Transportation       38.928234  36.056909   2.871325   
Enhanced-Ambition AFOFI                 0.381360   0.318181   0.063179   
                  Absortion / Removal        NaN  -6.927040        NaN   
                  Buildings            13.615448  12.756542   0.858906   
                  Electricity          65.004602  10.665352  54.339250   
                  Hydrogen              0.006011   0.082071  -0.076061   
                  Industry             55.471766  48.222087   7.249679   
                  Transportation       38.504139  34.187692   4.316447   

Year                                            rr  
scenario          sec                               
Current-Policy    AFOFI                  16.218777  
                  Buildings               0.474486  
                  Electricity            55.027089  
                  Hydrogen            -1285.064081  
                  Industry                6.704605  
                  Transportation          7.375946  
Enhanced-Ambition AFOFI                  16.566738  
                  Absortion / Removal          NaN  
                  Buildings               6.308323  
                  Electricity            83.592928  
                  Hydrogen            -1265.380793  
                  Industry               13.069134  
                  Transportation         11.210345

In [335]:
q = queries[265]
print(q.title)
dfCO2 = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
dfCO2['scenario'] = dfCO2['scenario'].str.split(',').str[0]
dfCO2['GHG'] = 'CO2'

CO2 emissions by tech (excluding resource production)


In [337]:
dfCO2[(dfCO2['sector'] == 'electricity')]

,Units,scenario,region,sector,subsector,technology,Year,value,GHG
433,MTC,Current-Policy,South Korea,electricity,coal,coal (conv pul ammonia blend 20%),2030,7.454610,CO2
434,MTC,Current-Policy,South Korea,electricity,coal,coal (conv pul ammonia blend 20%),2035,15.502170,CO2
435,MTC,Current-Policy,South Korea,electricity,gas,gas (CC H2 blend 50%),2025,0.169048,CO2
436,MTC,Current-Policy,South Korea,electricity,gas,gas (CC H2 blend 50%),2030,0.619459,CO2
437,MTC,Current-Policy,South Korea,electricity,gas,gas (CC H2 blend 50%),2035,1.306605,CO2
1800,MTC,Current-Policy-Chem,South Korea,electricity,coal,coal (conv pul ammonia blend 20%),2030,7.454610,CO2
1801,MTC,Current-Policy-Chem,South Korea,electricity,coal,coal (conv pul ammonia blend 20%),2035,15.502030,CO2
1802,MTC,Current-Policy-Chem,South Korea,electricity,gas,gas (CC H2 blend 50%),2025,0.169048,CO2
1803,MTC,Current-Policy-Chem,South Korea,electricity,gas,gas (CC H2 blend 50%),2030,0.619452,CO2
1804,MTC,Current-Policy-Chem,South Korea,electricity,gas,gas (CC H2 blend 50%),2035,1.306615,CO2


In [318]:
dfDiff.head(50)

Year                                                      2020          2035  \
scenario            sector                                                     
Current-Policy      ammonia                       2.195232e-01  4.556213e-01   
                    cement                        6.011810e+00  5.298641e+00   
                    chemical energy use           1.242345e+00  1.219657e+00   
                    chemical feedstocks           1.550905e-01 -1.055258e-03   
                    construction energy use       6.815726e-01  5.931784e-01   
                    construction feedstocks      -7.460139e-03 -1.165150e-02   
                    delivered biomass            -5.000001e-06 -5.999999e-06   
                    delivered gas                          NaN -4.763234e-09   
                    gas pipeline                 -4.688928e-07 -1.905295e-07   
                    gas processing                9.460443e-03  1.066423e-01   
                    iron and steel                1.961104e+01  1.553724e+01   
                    mining energy use             5.240690e-02  4.378410e-02   
                    other industrial energy use   1.535774e+01  1.670160e+01   
                    other industrial feedstocks  -4.473333e-03 -8.956290e-03   
                    process heat cement           3.128775e+00  3.223758e+00   
                    process heat food processing  5.083029e-01  5.218955e-01   
                    process heat paper            1.940351e-01  1.908125e-01   
                    refined liquids enduse                 NaN -1.309060e-07   
                    refining                      8.330040e+00  7.910167e+00   
                    waste biomass for paper      -5.377949e-05 -1.156587e-02   
                    wholesale gas                          NaN -4.167828e-08   
Current-Policy-Chem ammonia                       2.195232e-01  4.666352e-01   
                    cement                        6.011810e+00  5.291612e+00   
                    chemical energy use           1.242345e+00  1.244696e+00   
                    chemical feedstocks           1.550905e-01 -1.811882e+01   
                    construction energy use       6.815726e-01  5.991660e-01   
                    construction feedstocks      -7.460139e-03 -6.365810e-03   
                    delivered biomass            -5.000001e-06           NaN   
                    gas pipeline                 -4.688928e-07 -1.268816e-07   
                    gas processing                9.460443e-03  1.070721e-01   
                    iron and steel                1.961104e+01  1.561418e+01   
                    mining energy use             5.240690e-02  4.404020e-02   
                    other industrial energy use   1.535774e+01  1.683183e+01   
                    other industrial feedstocks  -4.473333e-03 -4.891896e-03   
                    process heat cement           3.128775e+00  3.306461e+00   
                    process heat food processing  5.083029e-01  5.266472e-01   
                    process heat paper            1.940351e-01  1.912484e-01   
                    refining                      8.330040e+00  6.638365e+00   
                    waste biomass for paper      -5.377949e-05 -1.156094e-02   
                    wholesale gas                          NaN -7.866634e-08   
Enhanced-Ambition   ammonia                       2.195240e-01  2.552682e-01   
                    cement                        6.012050e+00  4.988241e+00   
                    chemical energy use           1.242380e+00  1.184139e+00   
                    chemical feedstocks           1.601245e-01 -5.288523e-02   
                    construction energy use       6.816587e-01  5.913812e-01   
                    construction feedstocks      -7.326192e-03 -1.029430e-02   
                    delivered biomass                      NaN -4.000001e-06   
                    delivered gas                -5.215058e-09 -4.824520e-08   


In [283]:
55 * 3.66667

201.66684999999998

In [255]:
dfCO2Sec[(dfCO2Sec['var5'] == 'Emissions|CO2|Energy|Demand|Transportation')].groupby(['scenario', 'Year'])['value'].sum().tail(50)

scenario                        Year
EP-Power-Ind                    2015    37.439578
                                2020    39.144878
                                2025    39.137763
                                2030    36.475103
                                2035    33.311859
EP-Power-Ind-Trn                1975     1.652462
                                1990    13.410123
                                2005    33.846172
                                2010    34.386011
                                2015    37.439578
                                2020    38.491949
                                2025    36.569738
                                2030    34.277063
                                2035    34.060101
EP-Power-Ind-Trn-Bld            1975     1.652462
                                1990    13.410123
                                2005    33.846172
                                2010    34.386011
                                2015    37.439578
             

In [250]:
dfCO2Sec['var5'].unique()

array(['Emissions|CO2|Energy|Supply|Hydrogen',
       'Emissions|CO2|Energy|Demand|AFOFI',
       'Emissions|CO2|Energy|Demand|Industry',
       'Emissions|CO2|Energy|Supply|Electricity', nan,
       'Emissions|CO2|Energy|Demand|Residential and Commercial',
       'Emissions|CO2|Energy|Supply|Solids',
       'Emissions|CO2|Energy|Supply|Gases',
       'Emissions|CO2|Energy|Demand|Other Sector',
       'Emissions|CO2|Energy|Supply|Liquids',
       'Emissions|CO2|Energy|Demand|Transportation'], dtype=object)

In [229]:
dfCO2

,Units,scenario,region,sector,Year,value,GHG,sector_v7.1
0,MTC,EP-Power,South Korea,H2 central production,2020,1.684574e-03,CO2,others
1,MTC,EP-Power,South Korea,H2 central production,2025,1.089497e-01,CO2,others
2,MTC,EP-Power,South Korea,H2 central production,2030,5.522796e-01,CO2,others
3,MTC,EP-Power,South Korea,H2 central production,2035,1.174565e+00,CO2,others
4,MTC,EP-Power,South Korea,H2 wholesale dispensing,2025,1.025044e-01,CO2,others
...,...,...,...,...,...,...,...,...
4137,MTC,EP-Ref,South Korea,waste biomass for paper,2025,-2.780128e-04,CO2,others
4138,MTC,EP-Ref,South Korea,waste biomass for paper,2030,-2.727026e-03,CO2,others
4139,MTC,EP-Ref,South Korea,waste biomass for paper,2035,-1.114367e-02,CO2,others
4140,MTC,EP-Ref,South Korea,wholesale gas,2010,-3.020223e-07,CO2,others


In [224]:
dfCO2.groupby(['scenario', 'Year'])['value'].sum().tail(50)

scenario                        Year
EP-Power-Ind                    2015    181.474920
                                2020    173.911469
                                2025    169.116749
                                2030    128.769410
                                2035     87.733290
EP-Power-Ind-Trn                1975      9.621544
                                1990     72.647073
                                2005    144.682669
                                2010    169.797465
                                2015    181.474920
                                2020    173.289741
                                2025    166.705407
                                2030    126.316421
                                2035     87.513452
EP-Power-Ind-Trn-Bld            1975      9.621544
                                1990     72.647073
                                2005    144.682669
                                2010    169.797465
                                2015    181.4

In [228]:
dfCO2['sector'].unique()

array(['H2 central production', 'H2 wholesale dispensing',
       'agricultural energy use', 'ammonia', 'backup_electricity',
       'cement', 'chemical energy use', 'chemical feedstocks',
       'comm cooling', 'comm heating', 'comm others',
       'construction energy use', 'construction feedstocks',
       'delivered biomass', 'delivered gas', 'desalinated water',
       'elec_coal (IGCC CCS)', 'elec_coal (conv pul CCS)',
       'elec_gas (CC CCS)', 'electricity', 'gas pipeline',
       'gas processing', 'iron and steel', 'mining energy use',
       'other industrial energy use', 'other industrial feedstocks',
       'process heat cement', 'process heat food processing',
       'process heat paper', 'refined liquids enduse',
       'refined liquids industrial', 'refining', 'resid heating coal',
       'resid heating modern', 'resid others coal', 'resid others modern',
       'trn_aviation_intl', 'trn_freight', 'trn_freight_road', 'trn_pass',
       'trn_pass_road', 'trn_pass_road_LD

In [226]:
def cat_sector(sector: str):
    if sector == "cement":
        return "industry"
    elif sector.startswith('resid'):
        return "buildings"
    elif sector in ['buildings', 'electricity', 'industry', 'transportation']:
        return sector
    else:
        return "others"

In [227]:
dfCO2['sector_v7.1'] = dfCO2['sector'].apply(cat_sector)

In [146]:
dfCO2.head()

,Units,scenario,region,sector,Year,value,GHG,sector_v7.1
0,MTC,EP-Power,South Korea,FoodDemand_NonStaples,1990,0.269888,CO2,others
1,MTC,EP-Power,South Korea,FoodDemand_NonStaples,2005,0.295841,CO2,others
2,MTC,EP-Power,South Korea,FoodDemand_NonStaples,2010,0.274179,CO2,others
3,MTC,EP-Power,South Korea,FoodDemand_NonStaples,2015,0.277655,CO2,others
4,MTC,EP-Power,South Korea,FoodDemand_NonStaples,2020,0.291150,CO2,others


In [147]:
dfCO2[(dfCO2['scenario'] == 'EP-Power-Ind-Trn-Bld-Ag-Others')& (dfCO2['Year'].isin([2020, 2035]))].groupby(['scenario', 'Year', 'sector_v7.1'])['value'].sum().reset_index().head(50)

,scenario,Year,sector_v7.1,value
0,EP-Power-Ind-Trn-Bld-Ag-Others,2020,buildings,14.065744
1,EP-Power-Ind-Trn-Bld-Ag-Others,2020,electricity,65.110628
2,EP-Power-Ind-Trn-Bld-Ag-Others,2020,industry,51.220143
3,EP-Power-Ind-Trn-Bld-Ag-Others,2020,others,0.513929
4,EP-Power-Ind-Trn-Bld-Ag-Others,2020,transportation,42.076789
5,EP-Power-Ind-Trn-Bld-Ag-Others,2035,buildings,13.743860
6,EP-Power-Ind-Trn-Bld-Ag-Others,2035,electricity,10.543328
7,EP-Power-Ind-Trn-Bld-Ag-Others,2035,industry,85.599976
8,EP-Power-Ind-Trn-Bld-Ag-Others,2035,others,-6.282330
9,EP-Power-Ind-Trn-Bld-Ag-Others,2035,transportation,37.529735


In [148]:
dfCO2[(dfCO2['scenario'] == 'EP-Power-Ind-Trn-Bld-Ag-Others')& (dfCO2['Year'].isin([2020, 2035]))].groupby(['scenario', 'Year'])['value'].sum()

scenario                        Year
EP-Power-Ind-Trn-Bld-Ag-Others  2020    172.987232
                                2035    141.134569
Name: value, dtype: float64

In [74]:
q = queries[262]
print(q.title)
dfCO2Raw = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
dfCO2Raw['scenario'] = dfCO2Raw['scenario'].str.split(',').str[0]
dfCO2Raw['GHG'] = 'CO2'

CO2 emissions by sector (no bio) (excluding resource production)


In [153]:
industry = [
    'mining energy use', "aluminum", "agricultural energy use", "municipal water", 'desalinated water', "construction",
    "iron and steel", "other industry", "paper", "chemical", "N fertilizer", "cement"
]
dfCO2Raw[(dfCO2Raw['sector'].isin(industry)) & (dfCO2Raw['scenario'] == 'EP-Power-Ind-Trn-Bld-Ag-Others') & (dfCO2Raw['Year'].isin([2020, 2035]))]

,Units,scenario,region,sector,Year,value,GHG
2970,MTC,EP-Power-Ind-Trn-Bld-Ag-Others,South Korea,agricultural energy use,2020,0.381360,CO2
2973,MTC,EP-Power-Ind-Trn-Bld-Ag-Others,South Korea,agricultural energy use,2035,0.319578,CO2
2995,MTC,EP-Power-Ind-Trn-Bld-Ag-Others,South Korea,cement,2020,6.012050,CO2
2998,MTC,EP-Power-Ind-Trn-Bld-Ag-Others,South Korea,cement,2035,4.972744,CO2
3065,MTC,EP-Power-Ind-Trn-Bld-Ag-Others,South Korea,desalinated water,2020,0.000168,CO2
3068,MTC,EP-Power-Ind-Trn-Bld-Ag-Others,South Korea,desalinated water,2035,0.000281,CO2
3102,MTC,EP-Power-Ind-Trn-Bld-Ag-Others,South Korea,iron and steel,2020,19.611571,CO2
3105,MTC,EP-Power-Ind-Trn-Bld-Ag-Others,South Korea,iron and steel,2035,13.840143,CO2
3110,MTC,EP-Power-Ind-Trn-Bld-Ag-Others,South Korea,mining energy use,2020,0.052414,CO2
3113,MTC,EP-Power-Ind-Trn-Bld-Ag-Others,South Korea,mining energy use,2035,0.043969,CO2


In [150]:
dfCO2Raw['sector'].unique()

array(['H2 central production', 'H2 wholesale dispensing',
       'agricultural energy use', 'ammonia', 'backup_electricity',
       'cement', 'chemical energy use', 'chemical feedstocks',
       'comm cooling', 'comm heating', 'comm others',
       'construction energy use', 'construction feedstocks',
       'delivered biomass', 'delivered gas', 'desalinated water',
       'elec_coal (IGCC CCS)', 'elec_coal (conv pul CCS)',
       'elec_gas (CC CCS)', 'electricity', 'gas pipeline',
       'gas processing', 'iron and steel', 'mining energy use',
       'other industrial energy use', 'other industrial feedstocks',
       'process heat cement', 'process heat food processing',
       'process heat paper', 'refined liquids enduse',
       'refined liquids industrial', 'refining', 'resid heating coal_d1',
       'resid heating coal_d10', 'resid heating coal_d2',
       'resid heating coal_d3', 'resid heating coal_d4',
       'resid heating coal_d5', 'resid heating coal_d6',
       'resid he

In [ ]:
df

In [122]:
dfCO2[(dfCO2['technology'] == 'Soybean')]

,Units,scenario,region,sector,subsector,technology,Year,value,GHG
721,MTC,EP-Power-Ind-Trn-Bld-Ag-Others,South Korea,regional biomassOil,regional biomassOil,Soybean,2005,-0.009304,CO2
722,MTC,EP-Power-Ind-Trn-Bld-Ag-Others,South Korea,regional biomassOil,regional biomassOil,Soybean,2010,-0.281641,CO2
723,MTC,EP-Power-Ind-Trn-Bld-Ag-Others,South Korea,regional biomassOil,regional biomassOil,Soybean,2015,-0.353527,CO2
724,MTC,EP-Power-Ind-Trn-Bld-Ag-Others,South Korea,regional biomassOil,regional biomassOil,Soybean,2020,-0.421032,CO2
725,MTC,EP-Power-Ind-Trn-Bld-Ag-Others,South Korea,regional biomassOil,regional biomassOil,Soybean,2025,-0.400236,CO2
726,MTC,EP-Power-Ind-Trn-Bld-Ag-Others,South Korea,regional biomassOil,regional biomassOil,Soybean,2030,-0.316339,CO2
727,MTC,EP-Power-Ind-Trn-Bld-Ag-Others,South Korea,regional biomassOil,regional biomassOil,Soybean,2035,-0.277973,CO2


In [121]:
dfCO2['technology'].unique()

array(['biomass to H2', 'biomass to H2 CCS', 'coal chemical CCS',
       'gas ATR CCS', 'natural gas steam reforming', 'refined liquids',
       'biomass', 'gas', 'airCO2', 'gas CCS', 'hydrogen',
       'gas (steam/CT)', 'cement', 'cement LC3', 'biomass CCS', 'coal',
       'coal CCS', 'refined liquids CCS', 'distillation',
       'biomass (IGCC) (dry cooling)', 'biomass (IGCC) (recirculating)',
       'biomass (IGCC) (seawater)', 'biomass (conv) (dry cooling)',
       'biomass (conv) (once through)', 'biomass (conv) (recirculating)',
       'biomass (conv) (seawater)', 'coal (IGCC CCS) (dry cooling)',
       'coal (IGCC CCS) (recirculating)', 'coal (IGCC CCS) (seawater)',
       'coal (conv pul CCS) (dry cooling)',
       'coal (conv pul CCS) (once through)',
       'coal (conv pul CCS) (recirculating)',
       'coal (conv pul CCS) (seawater)', 'coal (conv pul) (dry cooling)',
       'coal (conv pul) (once through)',
       'coal (conv pul) (recirculating)', 'coal (conv pul) (seawater

In [87]:
q = queries[265]
print(q.title)
dfCO2Tech = conn.runQuery(q, scenarios=['EP-Power'], regions=['South Korea'])
dfCO2Tech['scenario'] = dfCO2Tech['scenario'].str.split(',').str[0]
dfCO2Tech['GHG'] = 'CO2'

CO2 emissions by tech (excluding resource production)


In [89]:
dfCO2.head()

,Units,scenario,region,sector,Year,value,GHG
0,MTC,EP-Power,South Korea,H2 central production,2020,0.001685,CO2
1,MTC,EP-Power,South Korea,H2 central production,2025,0.108950,CO2
2,MTC,EP-Power,South Korea,H2 central production,2030,0.552280,CO2
3,MTC,EP-Power,South Korea,H2 central production,2035,1.174565,CO2
4,MTC,EP-Power,South Korea,H2 wholesale dispensing,2025,0.102504,CO2


In [90]:
dfCO2Tech.head()

,Units,scenario,region,sector,subsector,technology,Year,value,GHG
0,MTC,EP-Power,South Korea,H2 central production,biomass,biomass to H2,2020,0.000290,CO2
1,MTC,EP-Power,South Korea,H2 central production,biomass,biomass to H2,2025,0.019025,CO2
2,MTC,EP-Power,South Korea,H2 central production,biomass,biomass to H2,2030,0.089967,CO2
3,MTC,EP-Power,South Korea,H2 central production,biomass,biomass to H2,2035,0.158684,CO2
4,MTC,EP-Power,South Korea,H2 central production,biomass,biomass to H2 CCS,2025,0.000005,CO2


In [91]:
dfCO2Tech.shape

(1372, 9)

In [102]:
dfCO2Comp = dfCO2.merge(dfCO2Tech, on=['Units', 'scenario', 'region', 'sector', 'Year', 'GHG'], how='outer', suffixes=['_tot', '_tech'])
dfCO2Comp

,Units,scenario,region,sector,Year,value_tot,GHG,subsector,technology,value_tech
0,MTC,EP-Power,South Korea,H2 central production,2020,1.684574e-03,CO2,biomass,biomass to H2,0.000290
1,MTC,EP-Power,South Korea,H2 central production,2020,1.684574e-03,CO2,gas,natural gas steam reforming,0.001681
2,MTC,EP-Power,South Korea,H2 central production,2025,1.089497e-01,CO2,biomass,biomass to H2,0.019025
3,MTC,EP-Power,South Korea,H2 central production,2025,1.089497e-01,CO2,biomass,biomass to H2 CCS,0.000005
4,MTC,EP-Power,South Korea,H2 central production,2025,1.089497e-01,CO2,coal,coal chemical CCS,0.001955
...,...,...,...,...,...,...,...,...,...,...
1407,MTC,EP-Power,South Korea,waste biomass for paper,2035,-1.114367e-02,CO2,biomass,biomass,0.510640
1408,MTC,EP-Power,South Korea,waste biomass for paper,2035,-1.114367e-02,CO2,biomass,biomass CCS,0.001238
1409,MTC,EP-Power,South Korea,waste biomass for paper,2035,-1.114367e-02,CO2,biomass,biomass cogen,0.115137
1410,MTC,EP-Power,South Korea,wholesale gas,2010,-3.020223e-07,CO2,NaN,NaN,NaN


In [103]:
dfCO2Comp[(dfCO2Comp['sector'].isna())]

,Units,scenario,region,sector,Year,value_tot,GHG,subsector,technology,value_tech


In [104]:
dfCO2Comp.sort_values(['sector', 'Year', 'subsector', 'technology']).head(50)

,Units,scenario,region,sector,Year,value_tot,GHG,subsector,technology,value_tech
0,MTC,EP-Power,South Korea,H2 central production,2020,0.001685,CO2,biomass,biomass to H2,0.000290
1,MTC,EP-Power,South Korea,H2 central production,2020,0.001685,CO2,gas,natural gas steam reforming,0.001681
2,MTC,EP-Power,South Korea,H2 central production,2025,0.108950,CO2,biomass,biomass to H2,0.019025
3,MTC,EP-Power,South Korea,H2 central production,2025,0.108950,CO2,biomass,biomass to H2 CCS,0.000005
4,MTC,EP-Power,South Korea,H2 central production,2025,0.108950,CO2,coal,coal chemical CCS,0.001955
5,MTC,EP-Power,South Korea,H2 central production,2025,0.108950,CO2,gas,gas ATR CCS,0.000019
6,MTC,EP-Power,South Korea,H2 central production,2025,0.108950,CO2,gas,natural gas steam reforming,0.106793
7,MTC,EP-Power,South Korea,H2 central production,2030,0.552280,CO2,biomass,biomass to H2,0.089967
8,MTC,EP-Power,South Korea,H2 central production,2030,0.552280,CO2,biomass,biomass to H2 CCS,0.000049
9,MTC,EP-Power,South Korea,H2 central production,2030,0.552280,CO2,coal,coal chemical CCS,0.010268


In [75]:
q = queries[262]
print(q.title)
dfCO2 = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
dfCO2['scenario'] = dfCO2['scenario'].str.split(',').str[0]
dfCO2['GHG'] = 'CO2'
q = queries[273]
print(q.title)
dfNonCO2 = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
dfNonCO2['scenario'] = dfNonCO2['scenario'].str.split(',').str[0]
dfGHG = pd.concat([dfCO2, dfNonCO2], axis=0)

CO2 emissions by sector (no bio) (excluding resource production)
nonCO2 emissions by tech (excluding resource production)


In [76]:
dfGHG['emiss(MT)'] = dfGHG.apply(to_Mt, axis=1)
dfGHG['gwpAr5'] = dfGHG['GHG'].apply(lambda gas: GWP_AR5[gas] if gas in GWP_AR5 else np.nan)
dfGHG['MTCO2eq'] = dfGHG['emiss(MT)'] * dfGHG['gwpAr5']

In [77]:
dfGHG.head()

,Units,scenario,region,sector,Year,value,GHG,subsector,technology,emiss(MT),gwpAr5,MTCO2eq
0,MTC,Current-Policy,South Korea,H2 central production,2020,0.003095,CO2,NaN,NaN,0.011340,1.0,0.011340
1,MTC,Current-Policy,South Korea,H2 central production,2025,0.007160,CO2,NaN,NaN,0.026234,1.0,0.026234
2,MTC,Current-Policy,South Korea,H2 central production,2030,0.006211,CO2,NaN,NaN,0.022758,1.0,0.022758
3,MTC,Current-Policy,South Korea,H2 central production,2035,0.009005,CO2,NaN,NaN,0.032995,1.0,0.032995
4,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2020,0.002876,CO2,NaN,NaN,0.010538,1.0,0.010538


In [78]:
dfGHG['sector'].unique()

array(['H2 central production', 'H2 wholesale dispensing',
       'agricultural energy use', 'ammonia', 'backup_electricity',
       'cement', 'chemical energy use', 'chemical feedstocks',
       'comm cooling', 'comm heating', 'comm others',
       'construction energy use', 'construction feedstocks',
       'delivered biomass', 'delivered gas', 'desalinated water',
       'elec_coal (IGCC CCS)', 'elec_coal (conv pul CCS)',
       'elec_gas (CC CCS)', 'electricity', 'gas pipeline',
       'gas processing', 'iron and steel', 'mining energy use',
       'other industrial energy use', 'other industrial feedstocks',
       'process heat cement', 'process heat food processing',
       'process heat paper', 'refined liquids enduse',
       'refined liquids industrial', 'refining', 'resid heating coal_d1',
       'resid heating coal_d10', 'resid heating coal_d2',
       'resid heating coal_d3', 'resid heating coal_d4',
       'resid heating coal_d5', 'resid heating coal_d6',
       'resid he

In [79]:
dfGHG[(dfGHG['sector'] == 'regional biomass')]

,Units,scenario,region,sector,Year,value,GHG,subsector,technology,emiss(MT),gwpAr5,MTCO2eq


In [35]:
dfGHG['Units'].unique()

array(['MTC', 'Gg', 'Tg'], dtype=object)

In [38]:
dfGHG[(dfGHG['Units'] == 'MTC')]['GHG'].unique()

array(['CO2', 'CO2_CCfD', 'CO2_Chemical', 'CO2_IronSteel'], dtype=object)

In [37]:
dfGHG['GHG'].unique()

array(['CO2', 'HFC125', 'HFC134a', 'HFC143a', 'HFC23', 'HFC32', 'SF6',
       'C2F6', 'CF4', 'HFC43', 'HFC227ea', 'HFC236fa', 'CO2_CCfD',
       'CO2_Chemical', 'CO2_IronSteel', 'CH4_AGR', 'N2O_AGR', 'NH3_AGR',
       'NMVOC_AGR', 'NOx_AGR', 'BC_AWB', 'CH4_AWB', 'CO_AWB', 'H2_AWB',
       'N2O_AWB', 'NH3_AWB', 'NMVOC_AWB', 'NOx_AWB', 'OC_AWB',
       'SO2_2_AWB', 'CH4', 'CO', 'NMVOC', 'NOx', 'SO2_2', 'H2', 'BC',
       'N2O', 'NH3', 'OC'], dtype=object)

In [ ]:
q = queries[262]
print(q.title)
dfCO2 = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
dfCO2['scenario'] = dfCO2['scenario'].str.split(',').str[0]
dfCO2['GHG'] = 'CO2'
q = queries[270]
print(q.title)
dfNonCO2 = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
dfNonCO2['scenario'] = dfNonCO2['scenario'].str.split(',').str[0]
dfGHG = pd.concat([dfCO2, dfNonCO2], axis=0)
dfGHG['emiss(MT)'] = dfGHG.apply(convert_to_mt, axis=1)
dfGHG['gwpAr5'] = dfGHG['GHG'].apply(lambda gas: gwpAr5[gas] if gas in gwpAr5 else np.nan)
dfGHG['MTCO2eq'] = dfGHG['emiss(MT)'] * dfGHG['gwpAr5']
dfGHG